
### KcBERT 기반 위험도 분류 모델(fine_tuning)

#### 🔻프로젝트 개요
- **목적**: 텍스트 메시지로부터 4단계 위험도 자동 분류
- **모델**: beomi/kcbert-base (한국어 특화 BERT)
- **분류 라벨**:
  - `positive` (보통): 정상 상태
  - `danger` (주의): 주의 필요
  - `critical` (위험): 위험 상태
  - `emergency` (경고): 긴급 상황

#### 🔻데이터 분할 전략
- **Train**: 60% (모델 학습)
- **Validation**: 20% (모델 선택 및 튜닝)
- **Test**: 20% (최종 성능 평가)
- **Stratified Split**: 모든 세트에서 클래스 비율 동일 유지

#### 🔻실행 순서
1. 라이브러리 설치 및 환경 설정
2. 데이터 로드 및 전처리
3. 데이터 분할
4. 모델 및 Dataset 정의
5. 학습 설정
6. 모델 학습
7. 최종 평가
8. 추론 함수 테스트  




---



### ⚙️ 환경 설정 및 저장 경로

In [ ]:
!pip install -r requirements.txt

In [ ]:
# ===========================================================================================
# 💻 코드 셀: 환경 설정
# ===========================================================================================

import pandas as pd
import numpy as np
import warnings

# 기본 라이브러리
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tqdm import tqdm

warnings.filterwarnings('ignore')

print("="*70)
print("환경 설정")
print("="*70)

# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n디바이스: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(f"⚠ GPU 미사용. CPU로 학습합니다.")

print("="*70 + "\n")

# ===========================================================================================
# 저장 경로 설정: 로컬(기본) 또는 Google Drive(선택)
# ===========================================================================================
import os 
BASE_DIR = os.getcwd()
# 저장 폴더 생성
os.makedirs(BASE_DIR, exist_ok=True)

# 실행 시간별 폴더

RUN_DIR = os.path.join(BASE_DIR, 'model')
RESULT_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
print(f"✓ 결과 저장 경로: {RUN_DIR}")

환경 설정

디바이스: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU 메모리: 42.47 GB

✓ 결과 저장 경로: /content/model


### 📊 1단계: 데이터 로드 및 전처리

#### 📁 데이터 파일
- **파일명**: `./data/(데이터라벨링)전체발화데이터(대전중구).csv`
- **위치**: 이 노트북과 같은 폴더
- **필수 컬럼**: `text`, `답변`

#### 🏷️ 라벨 매핑
| 한글 | 영문 | ID |
|------|------|-----|
| 보통 | positive | 0 |
| 주의 | danger | 1 |
| 위험 | critical | 2 |
| 경고 | emergency | 3 |

In [5]:
print("\n" + "="*70)
print("데이터 로드 및 전처리")
print("="*70)

# 데이터 로드
df = pd.read_excel('data/(데이터라벨링)전체발화데이터(대전중구).xlsx') # 파일 경로에 맞게 바꾸어야 함
print(f"✓ 원본 데이터: {len(df):,}개")

# 컬럼 추출 및 결측치 제거
df = df[['text', '답변']].dropna()
df.rename(columns={'답변':'label'},inplace=True)
print(f"✓ 전처리 후: {len(df):,}개")

# 숫자 라벨 생성
label2id = {'positive': 0, 'danger': 1, 'critical': 2, 'emergency': 3}
id2label = {v: k for k, v in label2id.items()}
df['label_id'] = df['label'].map(label2id)

# 분포 확인
print(f"\n라벨 분포:")
for label in ['positive', 'danger', 'critical', 'emergency']:
    count = df['label'].value_counts().get(label, 0)
    ratio = count / len(df) * 100
    print(f"  {label:<12}: {count:>6,}개 ({ratio:>5.2f}%)")


데이터 로드 및 전처리
✓ 원본 데이터: 32,928개
✓ 전처리 후: 32,928개

라벨 분포:
  positive    : 18,809개 (57.12%)
  danger      : 11,631개 (35.32%)
  critical    :  1,932개 ( 5.87%)
  emergency   :    556개 ( 1.69%)




---



### 🔀 2단계: 데이터 분할 (Stratified)

#### 🎯 Stratified Split
모든 세트에서 **클래스 비율을 동일하게 유지**

#### 📐 분할 과정
1. Train+Val (80%) vs Test (20%)
2. Train (60%) vs Val (20%)

In [6]:
print("\n" + "="*70)
print("데이터 분할")
print("="*70)

# Step 1: Train+Val vs Test
train_val_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label_id']
)

# Step 2: Train vs Val
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.25,
    random_state=42,
    stratify=train_val_df['label_id']
)

print(f"\n분할 결과:")
print(f"  Train: {len(train_df):>6,}개 ({len(train_df)/len(df)*100:>5.1f}%)")
print(f"  Val:   {len(val_df):>6,}개 ({len(val_df)/len(df)*100:>5.1f}%)")
print(f"  Test:  {len(test_df):>6,}개 ({len(test_df)/len(df)*100:>5.1f}%)")

# 클래스 분포
print(f"\n클래스 분포:")
print(f"{'Dataset':<10} {'positive':>10} {'danger':>10} {'critical':>10} {'emergency':>10}")
print("-" * 65)

for name, subset in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    counts = subset['label'].value_counts()
    total = len(subset)
    print(f"{name:<10} "
          f"{counts.get('positive', 0):>6} ({counts.get('positive', 0)/total*100:>5.1f}%) "
          f"{counts.get('danger', 0):>6} ({counts.get('danger', 0)/total*100:>5.1f}%) "
          f"{counts.get('critical', 0):>6} ({counts.get('critical', 0)/total*100:>5.1f}%) "
          f"{counts.get('emergency', 0):>6} ({counts.get('emergency', 0)/total*100:>5.1f}%)")

print("\n✓ 클래스 비율 일정하게 유지됨!")


데이터 분할

분할 결과:
  Train: 19,756개 ( 60.0%)
  Val:    6,586개 ( 20.0%)
  Test:   6,586개 ( 20.0%)

클래스 분포:
Dataset      positive     danger   critical  emergency
-----------------------------------------------------------------
Train       11285 ( 57.1%)   6978 ( 35.3%)   1159 (  5.9%)    334 (  1.7%)
Val          3762 ( 57.1%)   2327 ( 35.3%)    386 (  5.9%)    111 (  1.7%)
Test         3762 ( 57.1%)   2326 ( 35.3%)    387 (  5.9%)    111 (  1.7%)

✓ 클래스 비율 일정하게 유지됨!


### 🤖 3단계: Dataset 클래스 및 KcBERT 모델

#### Dataset 클래스
텍스트를 토큰화하고 PyTorch 텐서로 변환

#### KcBERT 모델
- 한국어 댓글로 학습된 BERT
- 110M 파라미터
- 4-way 분류 (positive/danger/critical/emergency)


In [7]:
# Dataset 클래스
class RiskDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=1024):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("✓ Dataset 클래스 정의 완료")

# KcBERT 모델 로드
print("\n모델 로드 중...")
MODEL_NAME = 'beomi/kcbert-base'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    problem_type="single_label_classification"
)
model.to(device)

print(f"✓ 모델 로드 완료: {sum(p.numel() for p in model.parameters()):,}개 파라미터")

# 하이퍼파라미터
BATCH_SIZE = 32
MAX_LENGTH = 128
EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

print(f"\n하이퍼파라미터:")
print(f"  Batch Size: {BATCH_SIZE}, Epochs: {EPOCHS}, LR: {LEARNING_RATE}")

✓ Dataset 클래스 정의 완료

모델 로드 중...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ 모델 로드 완료: 108,921,604개 파라미터

하이퍼파라미터:
  Batch Size: 32, Epochs: 5, LR: 2e-05


### ⚙️ 4단계: 데이터로더 및 학습 설정

#### DataLoader
배치 단위로 데이터 로드

#### Class Weight
불균형 데이터 처리 - 소수 클래스에 높은 가중치

#### Optimizer & Scheduler
- AdamW: Weight Decay 포함
- Linear Warmup + Decay

In [8]:
# 데이터로더 생성
train_dataset = RiskDataset(
    train_df['text'].reset_index(drop=True),
    train_df['label_id'].reset_index(drop=True),
    tokenizer, MAX_LENGTH
)
val_dataset = RiskDataset(
    val_df['text'].reset_index(drop=True),
    val_df['label_id'].reset_index(drop=True),
    tokenizer, MAX_LENGTH
)
test_dataset = RiskDataset(
    test_df['text'].reset_index(drop=True),
    test_df['label_id'].reset_index(drop=True),
    tokenizer, MAX_LENGTH
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"✓ DataLoader: Train {len(train_loader)}, Val {len(val_loader)}, Test {len(test_loader)} 배치")

# Class Weight
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2, 3]),
    y=train_df['label_id'].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"\nClass Weights:")
for label, weight in zip(['positive', 'danger', 'critical', 'emergency'], class_weights):
    print(f"  {label:<12}: {weight:.4f}")

# Optimizer & Scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

print(f"\n✓ Optimizer: AdamW")
print(f"✓ Total Steps: {total_steps:,}, Warmup: {warmup_steps:,}")


✓ DataLoader: Train 618, Val 206, Test 206 배치

Class Weights:
  positive    : 0.4377
  danger      : 0.7078
  critical    : 4.2614
  emergency   : 14.7874

✓ Optimizer: AdamW
✓ Total Steps: 3,090, Warmup: 309


### 🎓 5단계: 학습 및 평가 함수

#### train_epoch
- 1 에포크 학습 (Forward → Loss → Backward → Update)

#### eval_model
- 모델 평가 및 예측 결과 반환

In [9]:
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    losses = []
    correct_predictions = 0
    total_samples = 0

    progress_bar = tqdm(data_loader, desc='Training')

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        losses.append(loss.item())

        _, preds = torch.max(outputs.logits, dim=1)
        correct_predictions += torch.sum(preds == labels).item()
        total_samples += labels.size(0)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{correct_predictions/total_samples:.4f}'
        })

    return correct_predictions / total_samples, np.mean(losses)


def eval_model(model, data_loader, device):
    model.eval()
    predictions = []
    prediction_probs = []
    actual_labels = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating', leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            probs = torch.softmax(outputs.logits, dim=1)
            _, preds = torch.max(outputs.logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            prediction_probs.extend(probs.cpu().numpy())
            actual_labels.extend(labels.cpu().numpy())

    return predictions, prediction_probs, actual_labels

print("✓ 학습 및 평가 함수 정의 완료")


✓ 학습 및 평가 함수 정의 완료


### 🚀 6단계: 모델 학습

각 에포크마다:
1. Train set 학습
2. Validation set 평가
3. Best Model 저장 (Val F1 기준)

In [ ]:
print("\n" + "="*70)
print("모델 학습 시작")
print("="*70)

best_val_f1 = 0
history = {
    'train_loss': [],
    'train_acc': [],
    'val_f1_macro': [],
    'val_f1_weighted': []
}

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch + 1}/{EPOCHS}')
    print("="*70)

    # 학습
    train_acc, train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    print(f'[Train] Loss: {train_loss:.4f} | Acc: {train_acc:.4f}')

    # Validation
    val_predictions, val_probs, val_labels = eval_model(model, val_loader, device)
    val_f1_macro = f1_score(val_labels, val_predictions, average='macro')
    val_f1_weighted = f1_score(val_labels, val_predictions, average='weighted')
    history['val_f1_macro'].append(val_f1_macro)
    history['val_f1_weighted'].append(val_f1_weighted)
    print(f'[Val]   F1 Macro: {val_f1_macro:.4f} | F1 Weighted: {val_f1_weighted:.4f}')

    # 클래스별 F1
    val_f1_per_class = f1_score(val_labels, val_predictions, average=None)
    print("\n클래스별 F1:")
    for i, label in enumerate(['positive', 'danger', 'critical', 'emergency']):
        print(f'  {label:<12}: {val_f1_per_class[i]:.4f}')

    # Best Model 저장
    if val_f1_weighted > best_val_f1:
        best_val_f1 = val_f1_weighted
        model_path = os.path.join(RUN_DIR, 'best_kcbert_model.pt')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1_weighted': best_val_f1,
            'val_f1_macro': val_f1_macro
        }, model_path)
        print(f'\n✓ Best model saved! (Val F1: {best_val_f1:.4f})')

print("\n" + "="*70)
print("✓ 학습 완료!")
print("="*70)

# 히스토리 저장
history_df = pd.DataFrame(history)
history_df['epoch'] = range(1, len(history_df) + 1)
history_path = os.path.join(RESULT_DIR, 'training_history.csv')
history_df.to_csv(history_path, index=False, encoding='utf-8-sig')


모델 학습 시작

Epoch 1/5


Training: 100%|██████████| 618/618 [01:42<00:00,  6.04it/s, loss=0.7456, acc=0.6303]


[Train] Loss: 1.0940 | Acc: 0.6303


[Val]   F1 Macro: 0.5692 | F1 Weighted: 0.7295

클래스별 F1:
  positive    : 0.8343
  danger      : 0.6291
  critical    : 0.4055
  emergency   : 0.4080

✓ Best model saved! (Val F1: 0.7295)

Epoch 2/5


Training: 100%|██████████| 618/618 [01:40<00:00,  6.13it/s, loss=2.3386, acc=0.7544]


...

[Train] Loss: 0.2996 | Acc: 0.8704


[Val]   F1 Macro: 0.6286 | F1 Weighted: 0.7699

클래스별 F1:
  positive    : 0.8439
  danger      : 0.7153
  critical    : 0.4552
  emergency   : 0.5000

✓ Best model saved! (Val F1: 0.7699)

Epoch 5/5


Training: 100%|██████████| 618/618 [01:41<00:00,  6.11it/s, loss=0.2854, acc=0.9102]


[Train] Loss: 0.2021 | Acc: 0.9102


[Val]   F1 Macro: 0.6390 | F1 Weighted: 0.7745

클래스별 F1:
  positive    : 0.8491
  danger      : 0.7199
  critical    : 0.4431
  emergency   : 0.5438

✓ Best model saved! (Val F1: 0.7745)

✓ 학습 완료!


### 🎯 7단계: 최종 평가 (Test Set)

**학습에 전혀 사용되지 않은** Test Set으로 최종 성능 측정

In [ ]:
print("\n" + "="*70)
print("최종 평가 (Test Set)")
print("="*70)

# Best Model 로드
model_path = os.path.join(RUN_DIR, 'best_kcbert_model.pt')
checkpoint = torch.load(model_path)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"\n✓ Best Model (Epoch {checkpoint['epoch']+1})")
print(f"  Val F1: {checkpoint['val_f1_weighted']:.4f}")

# Test 평가
test_predictions, test_probs, test_labels = eval_model(model, test_loader, device)

test_f1_macro = f1_score(test_labels, test_predictions, average='macro')
test_f1_weighted = f1_score(test_labels, test_predictions, average='weighted')

print(f"\n{'='*70}")
print(f"🎯 Test F1 Macro:    {test_f1_macro:.4f}")
print(f"🎯 Test F1 Weighted: {test_f1_weighted:.4f}")
print(f"{'='*70}")

# Classification Report
target_names = ['positive', 'danger', 'critical', 'emergency']
print("\n[Classification Report]")
report_text = classification_report(
    test_labels,
    test_predictions,
    target_names=target_names,
    digits=4
)
print(report_text)

# Confusion Matrix
print("\n[Confusion Matrix]")
cm = confusion_matrix(test_labels, test_predictions)
cm_df = pd.DataFrame(
    cm,
    index=[f'True {l}' for l in target_names],
    columns=[f'Pred {l}' for l in target_names]
)
print(cm_df)

# 성능 비교
print(f"\n{'='*70}")
print("성능 비교")
print(f"{'='*70}")
print(f"Val F1:  {checkpoint['val_f1_weighted']:.4f}")
print(f"Test F1: {test_f1_weighted:.4f}")
print(f"차이:    {abs(checkpoint['val_f1_weighted'] - test_f1_weighted):.4f}")

if abs(checkpoint['val_f1_weighted'] - test_f1_weighted) < 0.03:
    verdict = "✓ 과적합 없음!"
elif checkpoint['val_f1_weighted'] > test_f1_weighted + 0.05:
    verdict = "⚠ 과적합 가능성"
else:
    verdict = "✓ 잘 학습됨!"

print(f"\n{verdict}")

# 결과 저장
report_path = os.path.join(RESULT_DIR, 'classification_report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)

cm_path = os.path.join(RESULT_DIR, 'confusion_matrix.csv')
cm_df.to_csv(cm_path, encoding='utf-8-sig')



최종 평가 (Test Set)

✓ Best Model (Epoch 5)
  Val F1: 0.7745



🎯 Test F1 Macro:    0.6016
🎯 Test F1 Weighted: 0.7682

[Classification Report]
              precision    recall  f1-score   support

    positive     0.8518    0.8386    0.8452      3762
      danger     0.7039    0.7309    0.7171      2326
    critical     0.4586    0.4005    0.4276       387
   emergency     0.3876    0.4505    0.4167       111

    accuracy                         0.7683      6586
   macro avg     0.6005    0.6051    0.6016      6586
weighted avg     0.7686    0.7683    0.7682      6586


[Confusion Matrix]
                Pred positive  Pred danger  Pred critical  Pred emergency
True positive            3155          535             50              22
True danger               493         1700            106              27
True critical              36          166            155              30
True emergency             20           14             27              50

성능 비교
Val F1:  0.7745
Test F1: 0.7682
차이:    0.0063

✓ 과적합 없음!


## 💡 8단계: 추론 함수

새로운 텍스트의 위험도를 예측하는 함수

In [12]:
def predict_risk(text, model, tokenizer, device):
    model.eval()

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
        pred = np.argmax(probs)

    predicted_label = id2label[pred]

    return {
        '위험도_등급': predicted_label,
        '신뢰도': float(probs[pred]),
        '전체_점수': {
            'positive': float(probs[0]),
            'danger': float(probs[1]),
            'critical': float(probs[2]),
            'emergency': float(probs[3])
        }
    }

print("✓ 추론 함수 정의 완료")

# 테스트
print("\n추론 예제:")
print("="*70)

test_texts = [
    "오늘 날씨가 좋네요",
    "우울해요",
    "죽고 싶어요",
    "도와줘"
]

for text in test_texts:
    result = predict_risk(text, model, tokenizer, device)
    print(f"\n'{text}'")
    print(f"→ {result['위험도_등급']} ({result['신뢰도']:.1%})")


✓ 추론 함수 정의 완료

추론 예제:

'오늘 날씨가 좋네요'
→ positive (99.7%)

'우울해요'
→ critical (80.3%)

'죽고 싶어요'
→ emergency (99.8%)

'도와줘'
→ emergency (99.8%)


In [13]:
# ===========================================================================================
# 💻 코드 셀: 최종 요약
# ===========================================================================================

print("\n" + "="*70)
print("🎉 모델 학습 및 평가 완료!")
print("="*70)
print(f"✓ 최종 Test F1: {test_f1_weighted:.4f}")
print(f"✓ 저장 위치: {RUN_DIR}")
print("="*70)


🎉 모델 학습 및 평가 완료!
✓ 최종 Test F1: 0.7682
✓ 저장 위치: /content/model


## 🎉 완료!

## 📁 저장된 파일
```
./result/
├── training_history.csv
├── classification_report.txt
└── confusion_matrix.csv

./model/
├── best_kcbert_model.pt
